# VisionAI — YOLOv8 Fine-Tuning on Videos + Images

**Supports:**
- 🎬 Videos (`.mp4`, `.avi`, `.mov`) → auto frame extraction
- 🖼️ Direct image folders (`.jpg`, `.png`, `.jpeg`) → used as-is
- 🚂 Train images, CCTV footage, night images — all supported

## Pipeline
```
Videos → extract frames every 2s  ┐
                                   ├─→ auto-label → fine-tune YOLOv8 → best.pt
Image folders → copy directly     ┘
```

## How to upload your data
**Option A — Upload button (top-right on this page):**
- Click ⬆ Upload → select all your .mp4 / .jpg files
- Set `VIDEO_DIR = '/kaggle/working/'`

**Option B — Kaggle Dataset (recommended for reuse):**
- Go to kaggle.com/datasets → New Dataset → upload files → name it
- Click '+ Add Input' on this notebook → search your dataset name → Add
- Set `VIDEO_DIR = '/kaggle/input/your-dataset-name/'`

## After training
Download `runs/detect/finetune/weights/best.pt` from Output tab
→ place in `backend/models/` → it appears in the ModelSelector dropdown.

In [ ]:
!pip install -q ultralytics opencv-python-headless

In [ ]:
import os, cv2, shutil, random, yaml
from pathlib import Path
from ultralytics import YOLO

# ══ CONFIGURE THESE ══════════════════════════════════════════════════════════

# ── Data sources ─────────────────────────────────────────────────────────────
# Folder with your .mp4/.avi videos AND/OR .jpg/.png image files.
# They can all be in the same folder — the notebook handles both.
VIDEO_DIR        = '/kaggle/working/'     # <-- change this to your upload path
IMAGE_DIRS       = []   # Extra image-only folders e.g. ['/kaggle/working/train_images/']
                         # Leave as [] if all data is in VIDEO_DIR

# ── Frame extraction (videos only) ───────────────────────────────────────────
FRAME_INTERVAL_S = 2    # 1 frame every N seconds from each video

# ── Pseudo-labelling model ────────────────────────────────────────────────────
# yolov8s.pt  → COCO 80 classes (person, car, train, bus, truck, bicycle ...)
# Use this if your images include trains, cars, people (ground-level)
PSEUDO_MODEL     = 'yolov8s.pt'
CONF_PSEUDO      = 0.20   # lower = more labels (better for night/partial views)

# ── Training ──────────────────────────────────────────────────────────────────
MODEL_TO_FINETUNE = 'yolov8s.pt'   # start from COCO weights
FINETUNE_EPOCHS  = 30
IMGSZ            = 640    # 640=fast, 1280=best accuracy (needs more VRAM)
BATCH            = 16     # reduce to 8 if Out-of-Memory error
TRAIN_SPLIT      = 0.85   # 85% train, 15% val
DATASET_ROOT     = '/kaggle/working/visionai_dataset'
# ═════════════════════════════════════════════════════════════════════════════
print('Config loaded ✅')

In [ ]:
# ── Step 1: Collect all frames (videos + images) ─────────────────────────────
frames_dir = Path(DATASET_ROOT) / 'raw_frames'
frames_dir.mkdir(parents=True, exist_ok=True)

total_frames = 0

# 1a. Extract frames from videos
video_exts = ['**/*.mp4', '**/*.avi', '**/*.mov', '**/*.mkv']
video_files = []
for ext in video_exts:
    video_files += list(Path(VIDEO_DIR).glob(ext))

print(f'Found {len(video_files)} video(s)')
for vid_path in video_files:
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    interval = max(1, int(fps * FRAME_INTERVAL_S))
    frame_idx, saved = 0, 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % interval == 0:
            name = f"{vid_path.stem}_f{frame_idx:06d}.jpg"
            cv2.imwrite(str(frames_dir / name), frame)
            saved += 1
        frame_idx += 1
    cap.release()
    print(f'  📹 {vid_path.name}: {saved} frames')
    total_frames += saved

# 1b. Copy direct images (from VIDEO_DIR and IMAGE_DIRS)
img_exts = ['**/*.jpg', '**/*.jpeg', '**/*.png']
all_image_dirs = [VIDEO_DIR] + IMAGE_DIRS
img_copied = 0
for src_dir in all_image_dirs:
    for ext in img_exts:
        for img_path in Path(src_dir).glob(ext):
            dst = frames_dir / f"img_{img_path.stem}_{img_copied}.jpg"
            if not dst.exists():
                shutil.copy(img_path, dst)
                img_copied += 1

total_frames += img_copied
print(f'\n🖼️  Direct images copied: {img_copied}')
print(f'📦 Total frames/images in dataset: {total_frames}')

In [ ]:
# ── Step 2: Auto-label everything with pseudo-label model ─────────────────────
print(f'Loading pseudo-label model: {PSEUDO_MODEL}')
pseudo_model = YOLO(PSEUDO_MODEL)

# Show what classes the model knows
print(f'Model knows {len(pseudo_model.names)} classes:')
print([pseudo_model.names[i] for i in range(min(20, len(pseudo_model.names)))])

labels_dir = Path(DATASET_ROOT) / 'raw_labels'
labels_dir.mkdir(exist_ok=True)

frame_files = sorted(frames_dir.glob('*.jpg'))
print(f'\nAuto-labelling {len(frame_files)} images at conf={CONF_PSEUDO}...')

labelled, skipped = 0, 0
for img_path in frame_files:
    results = pseudo_model.predict(str(img_path), conf=CONF_PSEUDO, verbose=False)
    r = results[0]
    label_lines = []
    for box in r.boxes:
        cls_id = int(box.cls[0])
        cx, cy, bw, bh = box.xywhn[0].tolist()
        label_lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    if label_lines:
        (labels_dir / (img_path.stem + '.txt')).write_text('\n'.join(label_lines))
        labelled += 1
    else:
        skipped += 1

print(f'✅ Labelled: {labelled} | ⬛ Skipped (no detections): {skipped}')
print(f'Tip: if too many skipped, lower CONF_PSEUDO (e.g. 0.10)')

In [ ]:
# ── Step 3: Build YOLO train/val split ───────────────────────────────────────
labelled_stems = {p.stem for p in labels_dir.glob('*.txt')}
labelled_imgs  = [p for p in frame_files if p.stem in labelled_stems]
random.shuffle(labelled_imgs)

split = int(len(labelled_imgs) * TRAIN_SPLIT)
splits = {'train': labelled_imgs[:split], 'val': labelled_imgs[split:]}
print(f'Train: {len(splits["train"])} images | Val: {len(splits["val"])} images')

if len(splits['val']) == 0:
    raise ValueError('Not enough labelled images for a val split. Add more data or lower CONF_PSEUDO.')

for split_name, imgs in splits.items():
    img_dir = Path(DATASET_ROOT) / split_name / 'images'
    lbl_dir = Path(DATASET_ROOT) / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img_path in imgs:
        shutil.copy(img_path, img_dir / img_path.name)
        lbl_path = labels_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            shutil.copy(lbl_path, lbl_dir / lbl_path.name)

class_names = list(pseudo_model.names.values())
data_yaml = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'val/images',
    'nc':    len(class_names),
    'names': class_names,
}
yaml_path = Path(DATASET_ROOT) / 'dataset.yaml'
yaml_path.write_text(yaml.dump(data_yaml))
print(f'\n✅ Dataset YAML written — {len(class_names)} classes')
print(f'   Includes train class: {"train" in class_names}')

In [ ]:
# ── Step 4: Fine-tune YOLOv8 ─────────────────────────────────────────────────
print(f'Fine-tuning {MODEL_TO_FINETUNE} for {FINETUNE_EPOCHS} epochs...')
model = YOLO(MODEL_TO_FINETUNE)

results = model.train(
    data=str(yaml_path),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,         # early stop if no improvement
    augment=True,        # mosaic + flips + colour jitter
    hsv_v=0.6,           # brightness variation → night/day robustness
    hsv_s=0.4,           # saturation variation
    degrees=5,           # small rotation (CCTV angle shifts)
    scale=0.4,           # scale jitter (objects at different distances)
    mixup=0.1,           # mixup augmentation (improves generalisation)
    project='/kaggle/working/runs/detect',
    name='finetune',
    exist_ok=True,
)
print('\n🎉 Training complete!')
print('Best model saved at: /kaggle/working/runs/detect/finetune/weights/best.pt')

In [ ]:
# ── Step 5: Evaluate + per-class results ─────────────────────────────────────
best_pt = '/kaggle/working/runs/detect/finetune/weights/best.pt'
finetuned = YOLO(best_pt)
val_results = finetuned.val(data=str(yaml_path), imgsz=IMGSZ, verbose=False)

print('\n── Overall Metrics ─────────────────────────────────')
print(f'mAP@0.5:      {val_results.box.map50*100:.2f}%')
print(f'mAP@0.5:0.95: {val_results.box.map*100:.2f}%')
print(f'Precision:    {val_results.box.mp*100:.2f}%')
print(f'Recall:       {val_results.box.mr*100:.2f}%')

# Per-class AP
print('\n── Per-Class AP@0.5 ────────────────────────────────')
ap_per_class = val_results.box.ap50   # array indexed by class id
names = finetuned.names
for cls_id, ap in enumerate(ap_per_class):
    if ap > 0:
        print(f'  {names.get(cls_id, cls_id):20s} {ap*100:.1f}%')

print('\n── Per-lighting evaluation ─────────────────────────')
import numpy as np
val_imgs = list((Path(DATASET_ROOT) / 'val' / 'images').glob('*.jpg'))
night, day = [], []
for p in val_imgs:
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    (night if img is not None and img.mean() < 60 else day).append(str(p))

def avg_det(img_list, label):
    if not img_list: print(f'{label}: no images'); return
    total = sum(len(finetuned.predict(p, conf=0.20, verbose=False)[0].boxes)
                for p in img_list[:40])
    print(f'{label} ({len(img_list)} imgs) avg detections/frame: {total/min(40,len(img_list)):.1f}')

avg_det(night, '🌙 Night')
avg_det(day,   '☀️  Day/Dusk')

print('\n✅ Download best.pt → Output tab → runs/detect/finetune/weights/best.pt')
print('   Place in backend/models/ → appears in ModelSelector automatically.')